# Encontro 5 · Gráficos honestos e imóveis parecidos

**Coleta e Preparação de Dados · PUCRS Online** — prática da Aula 5 (visualização, similaridade e dissimilaridade)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucaskup/pucrs-coleta-e-preparacao/blob/main/aula5/pratica/encontro5.ipynb)

Mesma base do Encontro 4 (Aurora Imóveis, dados sintéticos).

| Parte | Formato | Tempo |
| --- | --- | --- |
| A. Os gráficos do colega | em grupo: diagnosticar e refazer | 20 min |
| B. "Mostre imóveis parecidos com este" | cada grupo implementa **uma** medida de proximidade | 25 min |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = "https://lucaskup.github.io/pucrs-coleta-e-preparacao"
bruto = pd.read_csv(f"{BASE}/aula4/imoveis_aurora.csv")

# limpeza mínima (o tratamento completo é o tema do Encontro 6)
df = bruto.drop_duplicates().copy()
df["conservacao"] = df["conservacao"].str.lower().replace({"otimo": "ótimo"})
df["bairro"] = df["bairro"].replace({"Petropolis": "Petrópolis"})
df = df[(df["quartos"] >= 0) & (df["area_m2"].between(10, 600))]
df = df[df["codigo"] != "AUR-0060"].dropna(subset=["preco"])        # preço com um zero a mais
df["preco_m2"] = df["preco"] / df["area_m2"]
df = df.set_index("codigo")
print(df.shape)

## Parte A — Os gráficos do colega (20 min)

Para cada gráfico: (1) diga **qual pergunta** ele tenta responder; (2) aponte **o problema**
(canal visual, escala, eixo, cor, tipo de gráfico); (3) **refaça** na célula seguinte.

### A1. Preço médio por bairro

In [ ]:
# código do colega
media = df.groupby("bairro")["preco"].mean()
plt.figure(figsize=(9, 4))
plt.bar(media.index, media.values)
plt.ylim(600_000, media.max() * 1.02)
plt.xticks(rotation=90)
plt.title("Preço médio por bairro")
plt.show()

**Pergunta:** … **Problema:** … 

In [ ]:
# sua versão

### A2. Distribuição dos anúncios por bairro

In [ ]:
# código do colega
df["bairro"].value_counts().plot.pie(figsize=(6, 6), autopct="%.0f%%")
plt.ylabel("")
plt.show()

**Pergunta:** … **Problema:** … 

In [ ]:
# sua versão

### A3. Como se distribui a área dos imóveis?

In [ ]:
# código do colega
df["area_m2"].plot.hist(bins=3)
plt.show()

**Pergunta:** … **Problema:** … 

In [ ]:
# sua versão

### A4. Preço do m² por bairro

In [ ]:
# código do colega
med = df.groupby("bairro")["preco_m2"].median()
plt.plot(med.index, med.values, marker="o")
plt.xticks(rotation=90)
plt.show()

**Pergunta:** … **Problema:** … 

In [ ]:
# sua versão

### A5. Correlação entre atributos numéricos

In [ ]:
# código do colega
num = df[["preco", "area_m2", "quartos", "banheiros", "vagas", "iptu_anual", "preco_m2"]]
plt.imshow(num.corr(), cmap="jet")
plt.colorbar()
plt.show()

**Pergunta:** … **Problema:** … 

In [ ]:
# sua versão

---
## Parte B — "Mostre imóveis parecidos com este" (25 min)

O site quer exibir, na página do imóvel **AUR-0137** (apartamento de 3 quartos e 92 m² no Jardim
Botânico, R$ 897 mil, ótimo estado, 7 comodidades), os **5 anúncios mais parecidos**. Cada grupo
implementa **uma** medida e devolve o top-5. Na apresentação, comparamos as listas.

| Grupo | Medida | Atributos |
| --- | --- | --- |
| 1 | Euclidiana na escala original | `preco`, `area_m2`, `quartos` |
| 2 | Euclidiana após z-score | `preco`, `area_m2`, `quartos` |
| 3 | Jaccard | comodidades (binárias) |
| 4 | SMC (*simple matching*) | comodidades (binárias) |
| 5 | Gower | `tipo`, `bairro` (nominais), `conservacao` (ordinal), `preco`, `area_m2`, `quartos` (numéricos) |

Perguntas para o grupo responder antes de apresentar: *o seu top-5 faz sentido para quem está
procurando imóvel? Que tipo de "parecido" a sua medida captura, e o que ela ignora?*

In [ ]:
ALVO = "AUR-0137"
display(df.loc[[ALVO], ["tipo", "bairro", "preco", "area_m2", "quartos", "conservacao", "comodidades"]])

# comodidades como colunas 0/1 (útil para os grupos 3 e 4)
B = df["comodidades"].fillna("").str.get_dummies(sep=";")
print(B.shape)

def mostrar(dist, n=5):
    """Recebe uma Series de DISSIMILARIDADES indexada por código e mostra os n mais próximos do alvo."""
    top = dist.drop(ALVO).nsmallest(n)
    return df.loc[top.index, ["tipo", "bairro", "preco", "area_m2", "quartos", "conservacao"]].assign(dist=top.round(3))

### Grupo 1 — Euclidiana, escala original

In [ ]:
X = df[["preco", "area_m2", "quartos"]]
dist = ...
mostrar(dist)

### Grupo 2 — Euclidiana com z-score

In [ ]:
X = df[["preco", "area_m2", "quartos"]]
Z = ...
dist = ...
mostrar(dist)

### Grupo 3 — Jaccard nas comodidades

In [ ]:
def jaccard(x, y):
    ...
dist = 1 - B.apply(lambda linha: jaccard(linha, B.loc[ALVO]), axis=1)
mostrar(dist)

### Grupo 4 — SMC nas comodidades

In [ ]:
def smc(x, y):
    ...
dist = 1 - B.apply(lambda linha: smc(linha, B.loc[ALVO]), axis=1)
mostrar(dist)

### Grupo 5 — Gower (tipos mistos)

In [ ]:
ORDEM = {"precisa de reforma": 0, "regular": 1, "bom": 2, "ótimo": 3}

def gower(i, j):
    """Média das similaridades por atributo, cada uma em [0, 1]."""
    ...

dist = pd.Series({k: 1 - gower(k, ALVO) for k in df.index})
mostrar(dist)

### Comparação (para o mediador projetar na apresentação)
Cada grupo cola o seu top-5 abaixo. Quantos imóveis aparecem em mais de uma lista?

In [ ]:
# tops = {"euclid": [...], "zscore": [...], "jaccard": [...], "smc": [...], "gower": [...]}